In [ ]:
## all modules are complete
## directly calling from the module rag_pipeline now

# RAG Pipeline Quickstart

End-to-end demo of the modular `rag_pipeline` package.

This notebook is pure orchestration — all heavy code lives in `src/rag_pipeline/`.
Every cell is **idempotent**: re-running won't re-parse PDFs, re-embed chunks, or
duplicate work in the vectorstore.

Flow:
1. **Setup** — verify package install + config
2. **Ingest** — parse PDFs (skip if cached) → embed (skip if already in Chroma)
3. **Retriever** — assemble hybrid + reranker stack
4. **Demo** — ask one question, see the answer with citations
5. **Eval** — retrieval metrics on the eval set
6. **Safety** — refusal-rate check on negatives
7. **(Optional)** RAGAS scoring on a mini sample

In [ ]:
"""Verify the package install and show active config."""
from pathlib import Path
from rag_pipeline.config import cfg, log

log.info(f"Project root:   {cfg.PROJECT_ROOT}")
log.info(f"Provider:       {cfg.MODEL_PROVIDER}")
log.info(f"LLM:            {cfg.OLLAMA_MODEL}")
log.info(f"Embeddings:     {cfg.OLLAMA_EMBEDDING_MODEL}")
log.info(f"Top-K default:  {cfg.TOP_K}")

# Constants we'll reuse below
CHUNKS_CACHE = cfg.PROJECT_ROOT / "data" / "processed" / "phase1_chunks.json"
COLLECTION   = "IPC_Corpus"

In [ ]:
"""Parse raw docs → chunks → vectorstore. Skip whatever's already done.

REINDEX=True wipes the collection and starts fresh.  Use this exactly
once when migrating ID schemes (old random UUIDs → new content-hash).
After it succeeds, set REINDEX=False and forget it exists.
"""
from rag_pipeline.parsers import default_dispatcher, load_chunks_cache, save_chunks_cache
from rag_pipeline.vectorstore import get_vectorstore

REINDEX = True   # ← FIRST RUN: set True to wipe stale ID-scheme drift. Then set False.

# 1. Chunks — load from cache, parse only if missing
if CHUNKS_CACHE.exists():
    chunks = load_chunks_cache(CHUNKS_CACHE)
else:
    log.info("No chunks cache — parsing raw documents (slow, one-time)")
    dispatcher = default_dispatcher()
    chunks = dispatcher.parse_directory(cfg.DATA_RAW_DIR)
    save_chunks_cache(chunks, CHUNKS_CACHE)
log.info(f"Have {len(chunks)} chunks")

# 2. Vectorstore
vs = get_vectorstore(COLLECTION)

# Wipe if requested
if REINDEX:
    existing_before = vs._collection.count()
    log.warning(f"REINDEX=True → wiping {existing_before} vectors from '{COLLECTION}'")
    vs.delete_collection()
    # Clear the lru_cache so the next get_vectorstore() returns a fresh instance
    get_vectorstore.cache_clear()
    vs = get_vectorstore(COLLECTION)

# Idempotent ingest — embed only chunks NOT already indexed
existing_ids = set(vs.get(include=[])["ids"])
to_add = [c for c in chunks if c.chunk_id not in existing_ids]

if to_add:
    log.info(f"Embedding {len(to_add)} new chunks (existing: {len(existing_ids)})")
    docs = [c.to_langchain_document() for c in to_add]
    ids  = [c.chunk_id for c in to_add]
    BATCH = 64
    for i in range(0, len(to_add), BATCH):
        vs.add_documents(documents=docs[i:i + BATCH], ids=ids[i:i + BATCH])
    log.info(f"Vectorstore now has {vs._collection.count()} vectors")
else:
    log.info(f"Vectorstore already has all {len(existing_ids)} chunks — skipping ingest")

In [ ]:
"""Parse raw docs → chunks → vectorstore. Skip whatever's already done."""
from rag_pipeline.parsers import default_dispatcher, load_chunks_cache, save_chunks_cache
from rag_pipeline.vectorstore import get_vectorstore

# 1. Chunks — load from cache, or parse if cache is missing
if CHUNKS_CACHE.exists():
    chunks = load_chunks_cache(CHUNKS_CACHE)
else:
    log.info("No chunks cache — parsing raw documents (slow, one-time)")
    dispatcher = default_dispatcher()
    chunks = dispatcher.parse_directory(cfg.DATA_RAW_DIR)
    save_chunks_cache(chunks, CHUNKS_CACHE)
log.info(f"Have {len(chunks)} chunks")

# 2. Vectorstore — embed only chunks NOT already indexed
vs = get_vectorstore(COLLECTION)
existing_ids = set(vs.get(include=[])["ids"])
to_add = [c for c in chunks if c.chunk_id not in existing_ids]

if to_add:
    log.info(f"Embedding {len(to_add)} new chunks (existing: {len(existing_ids)})")
    docs = [c.to_langchain_document() for c in to_add]
    ids  = [c.chunk_id for c in to_add]
    BATCH = 64
    for i in range(0, len(to_add), BATCH):
        vs.add_documents(documents=docs[i:i + BATCH], ids=ids[i:i + BATCH])
    log.info(f"Vectorstore now has {vs._collection.count()} vectors")
else:
    log.info(f"Vectorstore already has all {len(existing_ids)} chunks — skipping ingest")

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

In [ ]:
from rag_pipeline.config import cfg
from rag_pipeline.eval import load_ragas_dataset, score_ragas_dataset

In [ ]:
# Reuse the dataset you saved in Part 2 of the Phase 2 work
rows = load_ragas_dataset(cfg.PROJECT_ROOT / "data" / "processed" / "ragas_rows_mq.json")[:2]
print(f"Scoring {len(rows)} rows...")

df = score_ragas_dataset(
    rows=rows,
    batch_size=2,
    checkpoint_path=cfg.PROJECT_ROOT / "eval" / "results" / "test_checkpoint.csv",
    output_path=cfg.PROJECT_ROOT / "eval" / "results" / "test_scored.csv",
)
print("\n✅ Scored:")
print(df[["user_input", "faithfulness", "answer_relevancy", "semantic_similarity"]].to_string(max_colwidth=40))